In [3]:
!pip install -U pip setuptools wheel
!pip install jupyter pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost lightgbm imbalanced-learn 
!pip install kagglehub

In [4]:
#To download datset from Kaggle:
import kagglehub
from pathlib import Path
import pandas as pd
path = kagglehub.dataset_download("dartweichen/student-life") 
print("Path to dataset files:", path)

Path to dataset files: /Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1


In [5]:
from pathlib import Path

base_path = Path(path)
print(base_path)
print(base_path.exists())

/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1
True


In [6]:
for p in base_path.rglob("*Stress*"):
    print(p)

/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/survey/PerceivedStressScale.csv
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u44.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u13.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u05.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u52.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u09.json
/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset/EMA/response/Stress/Stress_u25.json
/Users/leilatawfik/.

In [7]:
from pathlib import Path

stress_dir = base_path / "dataset" / "EMA" / "response" / "Stress"
print("stress_dir exists?", stress_dir.exists())
print("json count:", len(list(stress_dir.rglob("*.json"))))
print("csv count:", len(list(stress_dir.rglob("*.csv"))))

stress_dir exists? True
json count: 49
csv count: 0


In [8]:
import json

all_dfs = []
skipped = 0

for file in stress_dir.rglob("*.json"):
    try:
        text = file.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            skipped += 1
            continue

        try:
            obj = json.loads(text)
            df = pd.json_normalize(obj if isinstance(obj, list) else [obj])
        except json.JSONDecodeError:
            # JSON lines fallback
            rows = [json.loads(line) for line in text.splitlines() if line.strip()]
            df = pd.json_normalize(rows) if rows else pd.DataFrame()

        if df.empty:
            skipped += 1
            continue

        df["student_id"] = file.stem
        df["source_file"] = file.name
        all_dfs.append(df)

    except Exception:
        skipped += 1

stress_df = pd.concat(all_dfs, ignore_index=True, sort=False)

print("✅ stress_df shape:", stress_df.shape)
print("✅ participants:", stress_df["student_id"].nunique())
print("✅ skipped files:", skipped)
print("Columns:", list(stress_df.columns))
print(stress_df.head())

✅ stress_df shape: (2408, 6)
✅ participants: 49
✅ skipped files: 0
Columns: ['null', 'resp_time', 'level', 'location', 'student_id', 'source_file']
                       null   resp_time level location  student_id  \
0  43.70477575,-72.28844073  1364121982   NaN      NaN  Stress_u44   
1                         2  1364121983   NaN      NaN  Stress_u44   
2  43.70637091,-72.28704334  1364118696   NaN      NaN  Stress_u44   
3                         4  1364121980   NaN      NaN  Stress_u44   
4                         3  1364121985   NaN      NaN  Stress_u44   

       source_file  
0  Stress_u44.json  
1  Stress_u44.json  
2  Stress_u44.json  
3  Stress_u44.json  
4  Stress_u44.json  


In [9]:
stress_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2408 entries, 0 to 2407
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   null         240 non-null    str  
 1   resp_time    2408 non-null   int64
 2   level        2167 non-null   str  
 3   location     2167 non-null   str  
 4   student_id   2408 non-null   str  
 5   source_file  2408 non-null   str  
dtypes: int64(1), str(5)
memory usage: 113.0 KB


In [ ]:
# Number of missing values in 'level'
missing_count = stress_df["level"].isna().sum()
total_count = len(stress_df["level"])
print(total_count)

# Percentage of missing values in stress level:
print(missing_count/total_count * 100)

print("Missing values in 'level':", missing_count)

2408
10.008305647840531
Missing values in 'level': 241


In [60]:
def read_sensing_csv(filepath, encoding="utf-8", sniff_lines=50):
    """
    Reads sensing CSVs robustly.
    - If no trailing comma issue is detected, uses pd.read_csv normally.
    - If trailing commas / inconsistent trailing empty fields are detected,
      reads as text and normalizes each row to the header length.
    """
    # --- Detect trailing-comma pattern quickly (header or first N lines)
    with open(filepath, "r", encoding=encoding, errors="replace") as f:
        lines = f.read().splitlines()

    if not lines:
        return pd.DataFrame()

    header_line = lines[0]
    sample_lines = lines[1:1+sniff_lines]

    def has_trailing_comma(s: str) -> bool:
        return s.rstrip().endswith(",")

    suspicious = has_trailing_comma(header_line) or any(has_trailing_comma(s) for s in sample_lines)

    # --- If not suspicious, just do the normal pandas read
    if not suspicious:
        df = pd.read_csv(filepath)
        df.columns = df.columns.astype(str).str.strip()
        return df

    # --- Robust path: normalize row lengths to header length
    header = [h.strip() for h in header_line.split(",")]

    # remove trailing empty header fields caused by "...,"
    while header and header[-1] == "":
        header.pop()

    n_cols = len(header)
    if n_cols == 0:
        return pd.DataFrame()

    rows = []
    for line in lines[1:]:
        if not line.strip():
            continue

        parts = line.split(",")

        # If row has extra empty trailing fields due to commas at end, truncate
        if len(parts) > n_cols:
            parts = parts[:n_cols]
        # If row is short (rare but happens with malformed lines), pad
        elif len(parts) < n_cols:
            parts = parts + [""] * (n_cols - len(parts))

        rows.append(parts)

    df = pd.DataFrame(rows, columns=header)

    # Cleanup
    df.columns = df.columns.astype(str).str.strip()
    df = df.replace(r"^\s*$", pd.NA, regex=True)

    return df

In [61]:
sensing_root = base_path / "dataset" / "sensing"
sensing_feature_dfs = {}

for modality_folder in sensing_root.iterdir():
    if not modality_folder.is_dir():
        continue

    modality_name = modality_folder.name
    print(f"\nLoading {modality_name}...")

    all_dfs = []

    for file in modality_folder.glob("*.csv"):
        try:
            df = read_sensing_csv(file)

            if df.empty:
                continue

            df["student_id"] = file.stem
            all_dfs.append(df)

        except Exception as e:
            print(f"Skipping {file.name}: {e}")

    if all_dfs:
        sensing_feature_dfs[modality_name] = pd.concat(all_dfs, ignore_index=True)
        print(f"{modality_name} shape:", sensing_feature_dfs[modality_name].shape)
    else:
        print(f"No usable data for {modality_name}")

for name, df in sensing_feature_dfs.items():
    df.columns = df.columns.str.strip()
    sensing_feature_dfs[name] = df


Loading wifi...
wifi shape: (19244309, 5)

Loading gps...
gps shape: (202877, 11)

Loading activity...
activity shape: (22842191, 3)

Loading phonelock...
phonelock shape: (9275, 3)

Loading wifi_location...
wifi_location shape: (1893838, 3)

Loading audio...
audio shape: (99298223, 3)

Loading bluetooth...
bluetooth shape: (1288526, 5)

Loading dark...
dark shape: (7269, 3)

Loading phonecharge...
phonecharge shape: (3318, 3)

Loading conversation...
conversation shape: (79023, 3)


In [62]:
for name, df in sensing_feature_dfs.items():
    print("\n", name)
    print(df.columns)


 wifi
Index(['time', 'BSSID', 'freq', 'level', 'student_id'], dtype='str')

 gps
Index(['time', 'provider', 'network_type', 'accuracy', 'latitude', 'longitude',
       'altitude', 'bearing', 'speed', 'travelstate', 'student_id'],
      dtype='str')

 activity
Index(['timestamp', 'activity inference', 'student_id'], dtype='str')

 phonelock
Index(['start', 'end', 'student_id'], dtype='str')

 wifi_location
Index(['time', 'location', 'student_id'], dtype='str')

 audio
Index(['timestamp', 'audio inference', 'student_id'], dtype='str')

 bluetooth
Index(['time', 'MAC', 'class_id', 'level', 'student_id'], dtype='str')

 dark
Index(['start', 'end', 'student_id'], dtype='str')

 phonecharge
Index(['start', 'end', 'student_id'], dtype='str')

 conversation
Index(['start_timestamp', 'end_timestamp', 'student_id'], dtype='str')


In [ ]:
#Testing columns and data types after preprocessing:
for name, df in sensing_feature_dfs.items():
    print(f"\n{name}:")
    print(sensing_feature_dfs[name].head())


wifi:
                 time              BSSID  freq  level student_id  \
0 2013-03-27 04:00:03  00:0b:86:c1:29:01  2427    -94   wifi_u03   
1 2013-03-27 04:00:03  00:18:01:fb:74:d6  2462    -93   wifi_u03   
2 2013-03-27 04:00:03  00:1d:ce:7b:11:4d  2462    -90   wifi_u03   
3 2013-03-27 04:00:03  00:24:36:a7:61:bd  2427    -92   wifi_u03   
4 2013-03-27 04:00:03  20:aa:4b:cf:33:51  2437    -61   wifi_u03   

            timestamp        date  
0 2013-03-27 04:00:03  2013-03-27  
1 2013-03-27 04:00:03  2013-03-27  
2 2013-03-27 04:00:03  2013-03-27  
3 2013-03-27 04:00:03  2013-03-27  
4 2013-03-27 04:00:03  2013-03-27  

gps:
                 time provider network_type accuracy     latitude  \
0 2013-03-27 18:57:34  network         wifi   22.094   43.7066051   
1 2013-03-27 19:17:46  network         wifi   24.652   43.7065982   
2 2013-04-01 21:45:43  network         wifi    24.06    43.706614   
3 2013-04-01 22:05:42  network         wifi   25.242   43.7066032   
4 2013-04-01 22:0

In [68]:

for name, df in sensing_feature_dfs.items():
    if "timestamp" in df.columns:
        time_col = "timestamp"
    elif "time" in df.columns:
        time_col = "time"
    else:
        continue

    df = df.copy()

    # force numeric first (prevents NaT when values are strings like "1364410654")
    t = pd.to_numeric(df[time_col].astype(str).str.strip(), errors="coerce")
    df[time_col] = pd.to_datetime(t, unit="s", errors="coerce")

    # standardize column name
    df["timestamp"] = df[time_col]
    df["date"] = df["timestamp"].dt.date

    sensing_feature_dfs[name] = df

In [81]:
# Start and end timestamp column names:
start_candidates = [
    "start",
    "start_time",
    "start_timestamp",
]

end_candidates = [
    "end",
    "end_time",
    "end_timestamp",
]

def find_column(df, candidates):
    cand_set = {c.strip().lower() for c in candidates}
    for col in df.columns:
        if str(col).strip().lower() in cand_set:
            return col
    return None

def process_interval_df(df):
    start_col = find_column(df, start_candidates)
    end_col = find_column(df, end_candidates)

    if start_col is None or end_col is None:
        print("Start or end column not found.")
        return None

    df = df.copy()

    start_num = pd.to_numeric(df[start_col].astype(str).str.strip(), errors="coerce")
    end_num   = pd.to_numeric(df[end_col].astype(str).str.strip(), errors="coerce")

    df["start_dt"] = pd.to_datetime(start_num, unit="s", errors="coerce")
    df["end_dt"]   = pd.to_datetime(end_num, unit="s", errors="coerce")

    df["duration_s"] = (df["end_dt"] - df["start_dt"]).dt.total_seconds().clip(lower=0)
    df["date"] = df["start_dt"].dt.date

    return df

In [ ]:
start_endTimes = ["phonelock", "dark", "phonecharge", "conversation"]

for name in start_endTimes:
    df = sensing_feature_dfs[name]
    df_fixed = process_interval_df(df)

    if df_fixed is not None:
        sensing_feature_dfs[name] = df_fixed 
        print(f"{name} processed and saved.")
    else:
        print(f"{name} could not be processed.")

phonelock processed and saved.
dark processed and saved.
phonecharge processed and saved.
conversation processed and saved.


In [98]:
daily_feature_dfs = []

for name, df in sensing_feature_dfs.items():
    
    if "student_id" not in df.columns or "date" not in df.columns:
        continue

    df = df.copy()

    # If interval-based (has duration_s)
    if "duration_s" in df.columns:
        daily = (
            df.groupby(["student_id", "date"])["duration_s"]
            .sum()
            .reset_index()
        )
        daily = daily.rename(columns={"duration_s": f"{name}_duration_s"})
    
    else:
        # For point-based data, average numeric columns
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
        
        if len(numeric_cols) == 0:
            continue
        
        daily = (
            df.groupby(["student_id", "date"])[numeric_cols]
            .mean()
            .reset_index()
        )
        
        # rename columns to include modality name
        daily = daily.rename(columns={
            col: f"{name}_{col}" for col in numeric_cols
        })
    
    daily_feature_dfs.append(daily)

In [101]:
for k in ["gps", "wifi_location", "phonelock", "dark", "phonecharge", "conversation"]:
    df = sensing_feature_dfs[k]
    print("\n", k)
    print("columns:", df.columns.tolist())
    print("has date?", "date" in df.columns)
    print("has duration_s?", "duration_s" in df.columns)
    print("numeric cols:", df.select_dtypes(include="number").columns.tolist())


 gps
columns: ['time', 'provider', 'network_type', 'accuracy', 'latitude', 'longitude', 'altitude', 'bearing', 'speed', 'travelstate', 'student_id', 'timestamp', 'date']
has date? True
has duration_s? False
numeric cols: []

 wifi_location
columns: ['time', 'location', 'student_id', 'timestamp', 'date']
has date? True
has duration_s? False
numeric cols: []

 phonelock
columns: ['start', 'end', 'student_id', 'start_dt', 'end_dt', 'duration_s', 'date']
has date? True
has duration_s? True
numeric cols: ['start', 'end', 'duration_s']

 dark
columns: ['start', 'end', 'student_id', 'start_dt', 'end_dt', 'duration_s', 'date']
has date? True
has duration_s? True
numeric cols: ['start', 'end', 'duration_s']

 phonecharge
columns: ['start', 'end', 'student_id', 'start_dt', 'end_dt', 'duration_s', 'date']
has date? True
has duration_s? True
numeric cols: ['start', 'end', 'duration_s']

 conversation
columns: ['start_timestamp', 'end_timestamp', 'student_id', 'start_dt', 'end_dt', 'duration_s', '

In [99]:
from functools import reduce

full_daily_df = reduce(
    lambda left, right: pd.merge(
        left, right,
        on=["student_id", "date"],
        how="outer"
    ),
    daily_feature_dfs
)

In [100]:
print(full_daily_df.columns)

Index(['student_id', 'date', 'wifi_freq', 'wifi_level',
       'activity_activity inference', 'phonelock_duration_s',
       'audio_audio inference', 'bluetooth_class_id', 'bluetooth_level',
       'dark_duration_s', 'phonecharge_duration_s', 'conversation_duration_s'],
      dtype='str')
